In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_curve, auc

**Generating data**

In [71]:
np.random.seed(42)

X = np.random.randint(0, 2, size=(100000, 12))

Y = [(x[1] and x[4]) or (x[2] and x[8]) or (x[0] and x[11]) for x in X]

#Y = np.random.randint(0, 2, size=(100000, 1))

np.savetxt('dataIn.txt', X, fmt='%d')

np.savetxt('dataOut.txt', Y, fmt='%d')

**Loading data**

In [72]:
x = torch.tensor(np.loadtxt('dataIn.txt', delimiter=' ', dtype=int)).float()
y = torch.tensor(np.loadtxt('dataOut.txt', delimiter=' ', dtype=int))#.float()

train_size = int(len(x) * 0.8)

train_dataset = TensorDataset(x[:train_size], y[:train_size])
test_dataset = TensorDataset(x[train_size:], y[train_size:])

batch_size = 100

train_dataloader = DataLoader(train_dataset, batch_size = batch_size)
test_dataloader = DataLoader(test_dataset, batch_size = batch_size)

for x, y in test_dataloader:
    print(x.shape)
    print(y.shape)
    break

torch.Size([100, 12])
torch.Size([100])


**Declaring model**

In [73]:
#device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device = "cpu"
print(f"Using {device} device")

class Perceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin_stack = nn.Sequential(
            nn.Linear(12, 10),
            nn.LogSigmoid(),
            nn.Linear(10, 7),
            nn.LogSigmoid(),
            nn.Linear(7, 2)
        )

    def forward(self, x):
        y = self.lin_stack(x)
        return y

model = Perceptron().to(device)
print(model)

Using cpu device
Perceptron(
  (lin_stack): Sequential(
    (0): Linear(in_features=12, out_features=10, bias=True)
    (1): LogSigmoid()
    (2): Linear(in_features=10, out_features=7, bias=True)
    (3): LogSigmoid()
    (4): Linear(in_features=7, out_features=2, bias=True)
  )
)


**Train model**

In [74]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    print(len(dataloader))
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
            
        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

**Testing**

In [75]:
def test(dataloader, model):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)

            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

**Training**

In [77]:
epochs = 20

print(len(train_dataloader))
print(len(test_dataloader))

for i in range(epochs):
    print(f"Epoch {i+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model)

800
200
Epoch 1
-------------------------------
800
loss: 0.385964  [  100/80000]
loss: 0.290193  [10100/80000]
loss: 0.374850  [20100/80000]
loss: 0.318430  [30100/80000]
loss: 0.308522  [40100/80000]
loss: 0.334475  [50100/80000]
loss: 0.296624  [60100/80000]
loss: 0.254210  [70100/80000]
Test Error: Accuracy: 83.1%, Avg loss: 0.339849 

Epoch 2
-------------------------------
800
loss: 0.374357  [  100/80000]
loss: 0.275053  [10100/80000]
loss: 0.368900  [20100/80000]
loss: 0.310048  [30100/80000]
loss: 0.304634  [40100/80000]
loss: 0.323978  [50100/80000]
loss: 0.282580  [60100/80000]
loss: 0.248876  [70100/80000]
Test Error: Accuracy: 83.1%, Avg loss: 0.336586 

Epoch 3
-------------------------------
800
loss: 0.373141  [  100/80000]
loss: 0.270979  [10100/80000]
loss: 0.367008  [20100/80000]
loss: 0.307559  [30100/80000]
loss: 0.303138  [40100/80000]
loss: 0.319229  [50100/80000]
loss: 0.277239  [60100/80000]
loss: 0.246848  [70100/80000]
Test Error: Accuracy: 83.1%, Avg loss: 0

**Saving model**

In [9]:
#model.save(model.state_dict(), 'model.pth')